# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "patchtst_ohlcv_mse"
REPO_DIR = "/content/ECE1508_GenAI"   # absolute path -- see note below

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    # git -C targets REPO_DIR explicitly rather than `cd X && ...`, so this is correct
    # regardless of the kernel's current working directory when the cell reruns.
    #
    # fetch + checkout {BRANCH} (not just `pull`): REPO_DIR persists across notebook runs
    # within the same Colab VM session, so if an earlier run in this session cloned a
    # DIFFERENT branch (e.g. this VM was previously used for a notebook with an older
    # BRANCH value), a bare `pull` here would just pull more commits onto that stale
    # branch instead of switching -- and the push cell at the bottom would then fail with
    # "src refspec {BRANCH} does not match any", since no local branch by that name would
    # exist to push. Explicitly checking out BRANCH every time this cell runs avoids that.
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

# Absolute path, not "ECE1508_GenAI": os.path.isdir("ECE1508_GenAI") above is checked
# relative to the CURRENT working directory -- on a second run of this cell (after the
# %cd below already moved the kernel into /content/ECE1508_GenAI), that relative check
# looks for /content/ECE1508_GenAI/ECE1508_GenAI, finds nothing, and silently clones a
# second, nested copy of the repo inside the first one instead of pulling it.
%cd {REPO_DIR}


Cloning into '/content/ECE1508_GenAI'...
remote: Enumerating objects: 1913, done.
remote: Counting objects: 100% (1425/1425), done.
remote: Compressing objects: 100% (1024/1024), done.
remote: Total 1913 (delta 775), reused 1031 (delta 396), pack-reused 488 (from 1)
Receiving objects: 100% (1913/1913), 77.16 MiB | 19.71 MiB/s, done.
Resolving deltas: 100% (976/976), done.
/content/ECE1508_GenAI


In [3]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 6.8 MB/s eta 0:00:00


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [5]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: langsmith-0.10.2, typeguard-4.5.2, anyio-4.14.2
collected 30 items                                                             

steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [  3%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [  6%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [ 10%]
steven/tests/test_data_pipeline.py::test_build_window_shapes_and_masks PASSED [ 13%]
steven/tests/test_data_pipeline.py::test_to_patchtst_input_patch_padding_mask PASSED [ 16%]
steven/tests/test_data_pipeline.py::test_window_sampler_unique_and_within_bounds PASSED [ 20%]
steven/tests/test_data_pipeline.py::test_window_sampler_respects_split_boundary PASSED [ 23%]

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [ ]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [ ]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

## Evaluate both models on the fixed test set

In [ ]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [ ]:
!python steven/src/update_report.py

## Pull results back down

Zips `steven/outputs/` (checkpoints, metrics.json, sample_plots) and downloads it -- or just `git add`/`commit`/`push` from here if you'd rather sync back through the repo.

In [ ]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

## Backtest hf_patchtst_revin_no_volume (both channel_attention checkpoints)

Runs `steven/src/evaluate_revin.py`'s walk-forward backtest against both
`channel_attention` checkpoints from `train_patchtst_hf_channel_attention.ipynb`'s
`hf_patchtst_revin_no_volume` run (`steven/outputs/patchtst_revin_novolume_channel_attention_{false,true}_checkpoint.pt`)
-- this branch's best forecasting result so far, and the first checkpoint on this branch
with above-chance directional accuracy worth actually backtesting. See
`steven/src/evaluate_revin.py`'s module docstring for the full methodology (walk-forward,
take-profit-only, no CVAE). Writes `steven/outputs/backtest_<checkpoint filename>.json`
per checkpoint.


In [5]:
!python steven/src/evaluate_revin.py   --checkpoint steven/outputs/patchtst_revin_novolume_channel_attention_false_checkpoint.pt
!python steven/src/evaluate_revin.py   --checkpoint steven/outputs/patchtst_revin_novolume_channel_attention_true_checkpoint.pt


23:14:28 device: cuda
23:14:28 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
23:14:28 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
23:14:28 running walk-forward backtest (ctx=70 bars, confidence>=0.50, min_return>=0.100%, 24537..27006)...
23:14:33 walk-forward: 868 trades / 1379 decisions
23:14:33 confidence calibration (does confidence track correctness?):
23:14:33   {'bin': 'coherent-up (confidence=1)', 'n': 1246, 'win_rate': 0.7335473515248796, 'direction_accuracy': 0.5634028892455859}
23:14:33   {'bin': 'not coherent-up (confidence=0)', 'n': 133, 'win_rate': 0.40601503759398494, 'direction_accuracy': 0.42857142857142855}
23:14:33 wrote metrics to steven/outputs/backtest_patchtst_revin_novolume_channel_attention_false_checkpoint.json
23:14:33 walk_forward: {
  "checkpoint": "steven/outputs/patchtst_revin_novolume_channel_attention_false_checkpoint.pt",
  "checkpoint_config": {
    "model": {
      "channel_attention": fal

## Backtest hf_patchtst_revin_no_volume_patch14_14 (both channel_attention checkpoints)

Runs the same walk-forward backtest against the `patch_length=14, patch_stride=14` (no overlap) checkpoints -- a follow-up to `hf_patchtst_revin_no_volume_overlap` (`patch_length=14, patch_stride=7`, the current best backtest result on this branch: +9.68%/+12.81% total return). A wider patch-geometry sweep (lost to a Colab disconnect before it could be pushed) found this exact config nearly matched the overlap winner's RMSE gain while keeping dir acc above chance and coherence noticeably higher (0.937/0.946 vs. 0.927/0.901) -- suggesting patch length 14 itself, not overlap, was the real driver. This backtest checks whether that holds up in practice. Writes to `steven/outputs/backtest_<checkpoint filename>.json`, same as the backtests above.


In [ ]:
!python steven/src/evaluate_revin.py --checkpoint steven/outputs/patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.pt
!python steven/src/evaluate_revin.py --checkpoint steven/outputs/patchtst_revin_novolume_patch14_14_channel_attention_true_checkpoint.pt


## Backtest hf_patchtst_revin_no_volume_closeweighted (both channel_attention checkpoints)

Runs the same walk-forward backtest against the close-weighted-loss checkpoints (`CLOSE_WEIGHT=4.0`, open/high/low stay at 1.0, otherwise identical to step 4 -- `(7,7)` patches, 20 epochs, flat LR). Forecasting metrics for this run were flat-to-negative (dir acc roughly unchanged, coherence down, and close RMSE actually *worse* than the unweighted baseline despite the upweighting) -- but this branch has repeatedly shown forecasting metrics don't reliably predict backtest quality in either direction, so this checks directly rather than concluding from forecasting metrics alone. Writes to `steven/outputs/backtest_<checkpoint filename>.json`, same as the backtests above.


In [5]:
!python steven/src/evaluate_revin.py --checkpoint steven/outputs/patchtst_revin_novolume_closeweighted_channel_attention_false_checkpoint.pt
!python steven/src/evaluate_revin.py --checkpoint steven/outputs/patchtst_revin_novolume_closeweighted_channel_attention_true_checkpoint.pt


22:05:56 device: cuda
22:05:56 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
22:05:56 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
22:05:57 running walk-forward backtest (ctx=70 bars, confidence>=0.50, min_return>=0.100%, 24537..27006)...
22:06:03 walk-forward: 508 trades / 1805 decisions
22:06:03 confidence calibration (does confidence track correctness?):
22:06:03   {'bin': 'coherent-up (confidence=1)', 'n': 997, 'win_rate': 0.757271815446339, 'direction_accuracy': 0.555667001003009}
22:06:03   {'bin': 'not coherent-up (confidence=0)', 'n': 808, 'win_rate': 0.25123762376237624, 'direction_accuracy': 0.5457920792079208}
22:06:03 wrote metrics to steven/outputs/backtest_patchtst_revin_novolume_closeweighted_channel_attention_false_checkpoint.json
22:06:03 walk_forward: {
  "checkpoint": "steven/outputs/patchtst_revin_novolume_closeweighted_channel_attention_false_checkpoint.pt",
  "checkpoint_config": {
    "model": {
      

## Backtest without the coherence gate (`--confidence-threshold 0.0`)

Tests the simplified strategy: ignore coherence entirely, trade whenever `take_profit > close_0` and the predicted edge clears `min_return_threshold` (0.1%), on the two current best checkpoints (`patch14_14/False` and `overlap 14/7/True`). `--confidence-threshold` accepts any value in `(0.0, 1.0]` as "coherent-only" (the current default backtests use `0.5`); `0.0` additionally lets non-coherent windows trade. `confidence_calibration` has already shown non-coherent windows win only 25-41% of the time vs. 73-80% for coherent ones, so this is expected to dilute the win rate -- the open question is whether roughly doubling the trade count outweighs that on total return. Distinct `--metrics-out` paths so the existing coherence-gated results for these checkpoints aren't overwritten.


In [ ]:
!python steven/src/evaluate_revin.py --checkpoint steven/outputs/patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.pt --confidence-threshold 0.0 --metrics-out steven/outputs/backtest_patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint_noconf.json
!python steven/src/evaluate_revin.py --checkpoint steven/outputs/patchtst_revin_novolume_overlap_channel_attention_true_checkpoint.pt --confidence-threshold 0.0 --metrics-out steven/outputs/backtest_patchtst_revin_novolume_overlap_channel_attention_true_checkpoint_noconf.json


## Hysteresis-band backtest (Mamba-style entry/exit rule) on the best checkpoint

Runs `steven/src/evaluate_revin_hysteresis.py` -- a new, separate script (does not modify `evaluate_revin.py`) that ports the entry/exit rule used by this project's Mamba model to our OHLC-forecasting PatchTST checkpoint: enter long when the model's predicted bar-1 close return clears `+2bps`, stay long as long as it stays above `0bps`, exit to cash once it drops to/below `0bps`, never short. Unlike the take-profit strategy above, a position here can stay open for many hours -- there's no fixed 3-bar horizon and no take-profit limit order, the rule re-evaluates every hour off the model's predicted **bar-1** close return only (bars 2/3 of the forecast aren't used by this strategy). This first run uses the untuned defaults carried over from Mamba's own rule to establish a baseline before the thresholds are swept below.

Run on the current best checkpoint (`patch14_14`, `channel_attention=False`, +14.47%/+10.39% under the take-profit strategy) to see whether this different trading logic does better, worse, or about the same on the same model. Writes to `steven/outputs/backtest_hysteresis_<checkpoint filename>.json`.


In [ ]:
!python steven/src/evaluate_revin_hysteresis.py --checkpoint steven/outputs/patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.pt --enter-bps 2.0 --exit-bps 0.0 --cost-bps 0


## Hysteresis enter/exit threshold sweep (best checkpoint)

`enter_bps=2.0`/`exit_bps=0.0` above were carried over directly from Mamba's own tuned rule, not calibrated to this model's forecast distribution at all. `evaluate_revin_hysteresis.py --sweep` now supports sweeping this cheaply: the model forward pass runs exactly once (the expensive part), and every `(enter_bps, exit_bps)` combination is replayed against that cached forecast sequence with no further model calls -- so a 5x3 grid costs about the same as a single run. This sweep (and the finer one below) predates transaction-cost modeling, so both were run with `--cost-bps 0` to reproduce the originally-logged numbers unchanged; the script's *defaults* now include the 1bp-each-way cost (see the final costed backtest cell below).

Read the result looking for a **broad, stable region of good combinations**, not just the single highest-scoring cell -- with only one continuous test window (2024-01 to 2025-05), picking whichever grid cell happens to score highest risks curve-fitting the threshold to this one historical path rather than finding a genuinely better rule. `exit_bps >= enter_bps` combinations are skipped automatically (degenerate: no real "stay" band).


In [ ]:
!python steven/src/evaluate_revin_hysteresis.py --checkpoint steven/outputs/patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.pt --sweep --enter-bps-grid 1,2,3,4,5 --exit-bps-grid=-1,0,1 --cost-bps 0

## Finer hysteresis threshold sweep, centered on the coarse sweep's winner

The coarse 5x3 grid above found `enter_bps=4.0`/`exit_bps=-1.0` as the best combination (+30.86% total return, beating buy-and-hold for the first time on this branch), with two signs it's a real region rather than a lucky cell: `enter_bps=4` occupied the entire top 3 regardless of exit choice, and `exit_bps=0` (the untuned default) was the *worst* exit value at every `enter_bps` tested, consistently. This follow-up narrows `enter_bps` around 4 (`3.5`/`4.0`/`4.5`) and extends `exit_bps` further negative than the coarse grid tested (down to `-3`), since `-1` beat `0` and `1` consistently -- checking whether the true optimum sits even lower. Same cached-forecast-sequence approach, so this is still cheap regardless of grid size.


In [ ]:
!python steven/src/evaluate_revin_hysteresis.py --checkpoint steven/outputs/patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.pt --sweep --enter-bps-grid 3.5,4,4.5 --exit-bps-grid=-3,-2,-1,0,1 --cost-bps 0 --metrics-out steven/outputs/backtest_hysteresis_sweep_finer_patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.json


## Final costed hysteresis backtest (winning thresholds + 1bp transaction cost)

The finer sweep above found `enter_bps=4.5`/`exit_bps=-1.0` as the best combination (+31.35% total return / +22.09% annualized, the current best result on this branch), but that sweep ran with `--cost-bps 0` -- no transaction costs, matching how every earlier hysteresis result on this branch was measured. `evaluate_revin_hysteresis.py`'s defaults now bake in that winning `(4.5, -1.0)` configuration *and* a 1bp-on-entry + 1bp-on-exit transaction cost (`--cost-bps`, matching the reference Mamba rule this strategy was ported from -- see the module docstring), modeled as slippage on the fill price rather than a flat fee so it compounds correctly through the equity curve. This cell reruns that config with costs switched on, so the result is directly comparable to the (cost-free) sweep numbers above and shows how much of the edge survives realistic execution costs. Writes to a distinct `--metrics-out` path so the cost-free sweep results already logged in `steven/experiments.md` aren't overwritten.


In [5]:
!python steven/src/evaluate_revin_hysteresis.py --checkpoint steven/outputs/patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.pt --metrics-out steven/outputs/backtest_hysteresis_costed_patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.json


01:07:41 device: cuda
01:07:41 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
01:07:42 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
01:07:42 running hysteresis-band backtest (ctx=70 bars, enter>=4.5bps, exit<=-1.0bps, cost_bps=1.0, 24537..27006)...
01:07:50 hysteresis walk-forward: 158 trades / 2397 decisions
01:07:50 wrote metrics to steven/outputs/backtest_hysteresis_costed_patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.json
01:07:50 hysteresis_walk_forward: {
  "checkpoint": "steven/outputs/patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.pt",
  "checkpoint_config": {
    "model": {
      "channel_attention": false,
      "d_model": 64,
      "num_attention_heads": 4,
      "num_hidden_layers": 3,
      "dropout": 0.1,
      "head_dropout": 0.0,
      "patch_length": 14,
      "patch_stride": 14
    },
    "context_length": 70,
    "data_path": "steven/data/spy_ohlcv_1h.parquet",

### Commit + push backtest results

Same git identity/push pattern every other notebook in this family uses.


In [6]:
# Fresh Colab VM has no git identity configured -- needed for `commit` to work at all.
# Only sets it for this local clone (no --global), harmless to commit/share.
!git -C {REPO_DIR} config user.email "woodychang891121@gmail.com"
!git -C {REPO_DIR} config user.name "WoodyChang21"

!git add steven/outputs
!git status
!git commit -m "chore(model): log hf_patchtst_revin_no_volume walk-forward backtest results"


On branch patchtst_ohlcv_mse
Your branch is up to date with 'origin/patchtst_ohlcv_mse'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   steven/outputs/backtest_hysteresis_patchtst_revin_novolume_channel_attention_false_checkpoint.json
	new file:   steven/outputs/backtest_hysteresis_patchtst_revin_novolume_channel_attention_true_checkpoint.json
	new file:   steven/outputs/backtest_hysteresis_patchtst_revin_novolume_overlap_channel_attention_true_checkpoint.json
	new file:   steven/outputs/backtest_hysteresis_patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.json
	new file:   steven/outputs/backtest_hysteresis_sweep_patchtst_revin_novolume_patch14_14_channel_attention_false_checkpoint.json

[patchtst_ohlcv_mse 42162c8] chore(model): log hf_patchtst_revin_no_volume walk-forward backtest results
 5 files changed, 545 insertions(+)
 create mode 100644 steven/outputs/backtest_hysteresis_patchtst_revin_novolume_channel_attention_

In [7]:
import getpass

# Fresh Colab VM has no stored GitHub credentials, so a plain `git push` over HTTPS
# can't authenticate. Prompting interactively (getpass masks it, and it's never written
# into this notebook's saved source/outputs) instead of hardcoding a token in a cell --
# a hardcoded token would get committed into git history the moment this notebook is
# pushed, which is a real credential leak. Needs a GitHub Personal Access Token with
# `repo` scope: https://github.com/settings/tokens
token = getpass.getpass("GitHub Personal Access Token: ")
push_url = f"https://{token}@github.com/WoodyChang21/ECE1508_GenAI.git"
!git -C {REPO_DIR} push {push_url} {BRANCH}
del token, push_url  # don't leave it sitting in a notebook-visible variable longer than needed


To https://github.com/WoodyChang21/ECE1508_GenAI.git
 ! [rejected]        patchtst_ohlcv_mse -> patchtst_ohlcv_mse (fetch first)
error: failed to push some refs to 'https://github.com/WoodyChang21/ECE1508_GenAI.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.
